# Movie Recommendations

## Tasks:

- **Task 1:** EDA  
- **Task 2:** Data Wrangling / Feature Engineering  
- **Task 3:** Model Based Collaborative Filtering:
    - SVD  
    - NMF  
    - KNN  
- **Task 4:** Memory Based Collaborative Filtering:
    - User-based Collaborative Filtering  
    - Item-based Collaborative Filtering  
- **Task 5:** Report

---

## Task 1: EDA

Input dataset:  (MoviesLinks to an external site.) (RatingsLinks to an external site.)

Get to know your data (You know the drill by now)

---

## Task 2: Data Wrangling and Feature Engineering

- Check for outliers, missing data, null values, etc...  
- Can you come up with any new features?  
- Handle the 'genres' feature  
- Merge both data frames on movieId, and call this DataFrame, `Merged_df`  
- Create a new DataFrame to hold the User-Item Matrix data, called `UI_Matrix_df`

    ```text
    index = 'userId'
    columns = 'title'
    values = 'rating'
    ```

---

## Task 3: Model Based Collaborative Filtering

- Imputing (Predict) the missing rating data  
- Create three separate Model based Collaborative Filtering matrices. Ensure that each model's prediction doesn't interfere with the others.  
    - SVD  
    - NMF

---

## Task 4: Memory Based Collaborative Filtering

- Create a User-based Collaborative Filtering Method  
- Create two distinct User-User Matrices, one each from the Model based Collaborative Filtering matrices created task 3.  
- Populate the row-wise NaN's in `UI_Matrix_df` based on the User's Ratings  
- Try both Cosine and Pearson Similarities  
- Write a function to predict the Rating that a UserId (UID) might give for a given MovieId (MID)  
    - Rank order the similarity of UID to all other UserIds  
    - Predicted Rating = weighted avg. (based on similarity score) of k similar users' ratings of MID  
    - Test your function by predicting what User 1 might rate MovieId 32 based on the top 50 users.

- Create a Item-based Collaborative Filtering Method  
- Create two distinct Item-Item Matrices, one each from the Model based Collaborative Filtering matrices created in task 3.  
- Populate column-wise NaN's in `UI_Matrix_df` based on the Movie's Ratings  
- Try both Cosine and Pearson Similarities  
- Write a function to predict the top-N most similar movies to MovieId (MID) based on all ratings  
    - Rank order the similarity of MID to all other movies  
    - Return the top-N most similar movies  
    - Test your function by predicting what are the most similar movies to Jurassic Park (1993)

---

## Task 5: Report

You know the drill by now

---

## Task 6: Bonus Work (NOT REQUIRED)

- Create a Streamlit Dashboard to get Movie Recommendations for a user based on their liked Movies

In [22]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import KNNImputer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD, NMF, LatentDirichletAllocation


In [23]:
# Pull datasets
movies = pd.read_csv('../../../data/unit4/movies.csv')
ratings = pd.read_csv('../../../data/unit4/ratings.csv')

In [24]:
# Turn genres into list using '|' as a delimeter
movies['genres'] = movies['genres'].str.split('|')
# Remove genres = '(no genres listed)' from dataset and replace with an empty list
movies = movies[movies['genres'].apply(lambda x: '(no genres listed)' not in x)]
# look for null values
print(movies.info())
print(ratings.info())

<class 'pandas.core.frame.DataFrame'>
Index: 9708 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9708 non-null   int64 
 1   title    9708 non-null   object
 2   genres   9708 non-null   object
dtypes: int64(1), object(2)
memory usage: 303.4+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
None


In [25]:
# Pull year from title and create new column with the data
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
# Remove year from title
movies['title'] = movies['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)
# Print all rows that have a NaN year value
print(movies[movies['year'].isna()])
# Add the years to these movies manually
# There are lots of babylon 5 entries, we will just set the year for the first movie
movies.loc[movies['title'] == 'Babylon 5', 'year'] = 1993
# double checked all these against imdb
movies.loc[movies['title'] == 'Ready Player One', 'year'] = 2018
movies.loc[movies['title'] == 'Nocturnal Animals', 'year'] = 2016
movies.loc[movies['title'] == 'Moonlight', 'year'] = 2016
# Clean up any non unicode characters in title
movies['title'] = movies['title'].str.encode('utf-8', 'ignore').str.decode('utf-8')

      movieId              title                      genres  year
6059    40697          Babylon 5                    [Sci-Fi]   NaN
9031   140956   Ready Player One  [Action, Sci-Fi, Thriller]   NaN
9179   149334  Nocturnal Animals           [Drama, Thriller]   NaN
9367   162414          Moonlight                     [Drama]   NaN


In [26]:
# Merge datasets
movie_data = pd.merge(ratings, movies, on='movieId')
print(movie_data.head())
print(movie_data.info())

   userId  movieId  rating  timestamp                 title  \
0       1        1     4.0  964982703             Toy Story   
1       1        3     4.0  964981247      Grumpier Old Men   
2       1        6     4.0  964982224                  Heat   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en)   
4       1       50     5.0  964982931   Usual Suspects, The   

                                              genres    year  
0  [Adventure, Animation, Children, Comedy, Fantasy]  1995.0  
1                                  [Comedy, Romance]  1995.0  
2                          [Action, Crime, Thriller]  1995.0  
3                                [Mystery, Thriller]  1995.0  
4                         [Crime, Mystery, Thriller]  1995.0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100789 entries, 0 to 100788
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100789 non-null  int64  
 1   movie

In [27]:
# Create User-Item matrix df using the pivot_table method
ui_matrix_df = movie_data.pivot_table(index='userId', columns='title', values='rating')


In [28]:
# Create a collab filter matrix by filling in missing values with zeros
ui_matrix_filled_df = ui_matrix_df.fillna(0)

# Create a TruncatedSVD model to reduce dimensionality and group similar movies together
svd = TruncatedSVD(n_components=50, random_state=691)
svd_model = svd.fit_transform(ui_matrix_filled_df)
svd_matrix = np.dot(svd_model, svd.components_)
svd_df = pd.DataFrame(svd_matrix, index=ui_matrix_df.index, columns=ui_matrix_df.columns)
print(svd_df.head())

# Create a NMF model to reduce dimensionality and group similar movies together
nmf = NMF(n_components=50, random_state=691)
nmf_model = nmf.fit_transform(ui_matrix_filled_df)
nmf_matrix = np.dot(nmf_model, nmf.components_)
nmf_df = pd.DataFrame(nmf_matrix, index=ui_matrix_df.index, columns=ui_matrix_df.columns)
print(nmf_df.head())

# Create a LDA model to reduce dimensionality and group similar movies together
lda = LatentDirichletAllocation(n_components=50, random_state=691, n_jobs=-1)
lda_model = lda.fit_transform(ui_matrix_filled_df)
lda_matrix = np.dot(lda_model, lda.components_)
lda_df = pd.DataFrame(lda_matrix, index=ui_matrix_df.index, columns=ui_matrix_df.columns)
print(lda_df.head())

title        '71  'Hellboy': The Seeds of Creation  'Round Midnight  \
userId                                                                
1      -0.080478                          0.044068        -0.023611   
2      -0.029492                         -0.006939        -0.008001   
3       0.018744                          0.000118         0.002377   
4      -0.009614                          0.007801         0.003026   
5       0.015745                          0.000211        -0.004552   

title   'Salem's Lot  'Til There Was You  'Tis the Season for Love  \
userId                                                               
1          -0.040178           -0.048289                  0.010964   
2          -0.002711           -0.003529                 -0.005204   
3           0.001228            0.000467                 -0.000212   
4           0.012480            0.023781                 -0.031603   
5          -0.007157           -0.010151                 -0.005807   

title   'bu

f:\AI and Machine Learning course\FSA_devops\.venv-13\Lib\site-packages\sklearn\decomposition\_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


title       '71  'Hellboy': The Seeds of Creation  'Round Midnight  \
userId                                                               
1       0.00000                          0.018622         0.022262   
2       0.00000                          0.001755         0.001361   
3       0.00294                          0.000123         0.002406   
4       0.00000                          0.016948         0.021999   
5       0.00000                          0.004757         0.004492   

title   'Salem's Lot  'Til There Was You  'Tis the Season for Love  \
userId                                                               
1           0.001154            0.001164                  0.001969   
2           0.000070            0.000020                  0.000464   
3           0.001360            0.001333                  0.000000   
4           0.009322            0.033318                  0.000583   
5           0.000797            0.003195                  0.000000   

title   'burbs, Th

In [29]:
# Create a user to user matrix for each of the three models to find similar users
svd_user_sim = pd.DataFrame(cosine_similarity(svd_df), index=svd_df.index, columns=svd_df.index)
nmf_user_sim = pd.DataFrame(cosine_similarity(nmf_df), index=nmf_df.index, columns=nmf_df.index)
lda_user_sim = pd.DataFrame(cosine_similarity(lda_df), index=lda_df.index, columns=lda_df.index)
# Create 3 more using Pearson similarity instead of cosine similarity
svd_user_sim_pearson = svd_df.T.corr(method='pearson')
nmf_user_sim_pearson = nmf_df.T.corr(method='pearson')
lda_user_sim_pearson = lda_df.T.corr(method='pearson')
# Print these user to user similarity matrices
print("SVD User Similarity (Cosine):")
print(svd_user_sim.head())
print("SVD User Similarity (Pearson):")
print(svd_user_sim_pearson.head())
print("NMF User Similarity (Cosine):")
print(nmf_user_sim.head())
print("NMF User Similarity (Pearson):")
print(nmf_user_sim_pearson.head())
print("LDA User Similarity (Cosine):")
print(lda_user_sim.head())
print("LDA User Similarity (Pearson):")
print(lda_user_sim_pearson.head())

SVD User Similarity (Cosine):
userId       1         2         3         4         5         6         7    \
userId                                                                         
1       1.000000 -0.007674  0.405362  0.496114  0.312159  0.199020  0.360642   
2      -0.007674  1.000000 -0.007516 -0.035207  0.041669  0.063560  0.071631   
3       0.405362 -0.007516  1.000000 -0.053264  0.028869  0.041052  0.073609   
4       0.496114 -0.035207 -0.053264  1.000000  0.344278  0.181878  0.282134   
5       0.312159  0.041669  0.028869  0.344278  1.000000  0.770358  0.296063   

userId       8         9         10   ...       601       602       603  \
userId                                ...                                 
1       0.251614  0.203952 -0.039727  ...  0.171514  0.302437  0.339297   
2       0.087993  0.012167  0.350088  ...  0.620571  0.063119  0.035233   
3       0.016788 -0.005549 -0.001698  ...  0.008605  0.063252  0.217083   
4       0.211232  0.275968  0.1189

In [30]:
# Create an item to item matrix for each of the three models to find similar movies
svd_item_sim = pd.DataFrame(cosine_similarity(svd_df.T), index=svd_df.columns, columns=svd_df.columns)
nmf_item_sim = pd.DataFrame(cosine_similarity(nmf_df.T), index=nmf_df.columns, columns=nmf_df.columns)
lda_item_sim = pd.DataFrame(cosine_similarity(lda_df.T), index=lda_df.columns, columns=lda_df.columns)
# Create 3 more using Pearson similarity instead of cosine similarity
svd_item_sim_pearson = svd_df.corr(method='pearson')
nmf_item_sim_pearson = nmf_df.corr(method='pearson')
lda_item_sim_pearson = lda_df.corr(method='pearson')
# Print these item to item similarity matrices
print("SVD Item Similarity (Cosine):")
print(svd_item_sim.head())
print("SVD Item Similarity (Pearson):")
print(svd_item_sim_pearson.head())
print("NMF Item Similarity (Cosine):")
print(nmf_item_sim.head())
print("NMF Item Similarity (Pearson):")
print(nmf_item_sim_pearson.head())
print("LDA Item Similarity (Cosine):")
print(lda_item_sim.head())
print("LDA Item Similarity (Pearson):")
print(lda_item_sim_pearson.head())

SVD Item Similarity (Cosine):
title                                  '71  'Hellboy': The Seeds of Creation  \
title                                                                          
'71                               1.000000                         -0.012843   
'Hellboy': The Seeds of Creation -0.012843                          1.000000   
'Round Midnight                  -0.003017                          0.899465   
'Salem's Lot                      0.063778                          0.400703   
'Til There Was You                0.085153                          0.254475   

title                             'Round Midnight  'Salem's Lot  \
title                                                             
'71                                     -0.003017      0.063778   
'Hellboy': The Seeds of Creation         0.899465      0.400703   
'Round Midnight                          1.000000      0.679190   
'Salem's Lot                             0.679190      1.000000   
'Til Th

In [31]:
# Write helper functions to convert movie id to movie titles and vice-versa
def movie_id_to_title(movie_id):
    title = movies.loc[movies['movieId'] == movie_id, 'title']
    if not title.empty:
        return title.values[0]
    else:
        return None
def movie_title_to_id(title):
    movie_id = movies.loc[movies['title'] == title, 'movieId']
    if not movie_id.empty:
        return movie_id.values[0]
    else:
        return None

# Write functions to predict ratings for a user based on a given movie title or movie ID
# using the item similarity matrix and user similarity matrices averaging k amount of similar users' ratings
# also choose whether cosine or pearson similarity is used by passing in 'cosine' or 'pearson' as the similarity metric
# Return the prediction for the user based on the specified model and similarity metric and the top 5 similar movies
# as a tuple (predicted_rating, similar_movies)
def predict_rating_item_based(user_id, movie_title, model='svd', similarity_metric='cosine', k=10, n=5):
    if model == 'svd':
        if similarity_metric == 'cosine':
            item_sim_matrix = svd_item_sim
        else:
            item_sim_matrix = svd_item_sim_pearson
    elif model == 'nmf':
        if similarity_metric == 'cosine':
            item_sim_matrix = nmf_item_sim
        else:
            item_sim_matrix = nmf_item_sim_pearson
    else:
        if similarity_metric == 'cosine':
            item_sim_matrix = lda_item_sim
        else:
            item_sim_matrix = lda_item_sim_pearson

    if movie_title not in item_sim_matrix.columns:
        return None  # Movie not found in similarity matrix

    # Get similarity scores for the given movie
    sim_scores = item_sim_matrix[movie_title].dropna()
    sim_scores = sim_scores.sort_values(ascending=False)
    # Get the top k similar movies
    top_k_movies = sim_scores.index[1:k+1]  # Exclude the movie itself
    top_k_scores = sim_scores.values[1:k+1]
    # Get the user's ratings for these movies
    user_ratings = ui_matrix_df.loc[user_id, top_k_movies]
    # Calculate the predicted rating as a weighted average
    if user_ratings.isnull().all():
        return None  # User has not rated any of the similar movies
    weighted_ratings = user_ratings.fillna(0) * top_k_scores
    predicted_rating = weighted_ratings.sum() / np.sum(top_k_scores[user_ratings.notnull()])
    return (predicted_rating, top_k_movies.tolist()[:n])  # Return top n similar movies



In [32]:
# Test the functions
from turtle import mode


user_id = 1
movie_title = 'Jurassic Park'
predicted_rating, similar_movies = predict_rating_item_based(user_id, movie_title, model='svd', similarity_metric='cosine', k=10)
print("model = svd, similarity_metric = cosine")
print(f"Predicted rating for user {user_id} on movie '{movie_title}': {predicted_rating}")
print(f"Top similar movies to '{movie_title}': {similar_movies}")
predicted_rating, similar_movies = predict_rating_item_based(user_id, movie_title, model='lda', similarity_metric='pearson', k=10)
print("model = lda, similarity_metric = pearson")
print(f"Predicted rating for user {user_id} on movie '{movie_title}': {predicted_rating}")
print(f"Top similar movies to '{movie_title}': {similar_movies}")
predicted_rating, similar_movies = predict_rating_item_based(user_id, movie_title, model='nmf', similarity_metric='pearson', k=10)
print("model = nmf, similarity_metric = pearson")
print(f"Predicted rating for user {user_id} on movie '{movie_title}': {predicted_rating}")
print(f"Top similar movies to '{movie_title}': {similar_movies}")

model = svd, similarity_metric = cosine
Predicted rating for user 1 on movie 'Jurassic Park': 4.20257482611742
Top similar movies to 'Jurassic Park': ['Terminator 2: Judgment Day', 'Speed', 'Fugitive, The', 'Batman', 'Braveheart']
model = lda, similarity_metric = pearson
Predicted rating for user 1 on movie 'Jurassic Park': 4.168102841552428
Top similar movies to 'Jurassic Park': ['Fugitive, The', 'Batman', 'Mask, The', 'Desperado', 'GoldenEye']
model = nmf, similarity_metric = pearson
Predicted rating for user 1 on movie 'Jurassic Park': 4.2500580357890865
Top similar movies to 'Jurassic Park': ['Terminator 2: Judgment Day', 'Speed', 'Braveheart', 'Batman', 'Fugitive, The']


### Report
- First, organizing the data and filling in lost data like years of the films was necessary to make the data set the same all around.
- Next, creating and organizing the matrices were a task that did not make much sense to me.
   - Realizing that the key to making them was the directionality of the pivot table was the main breakthrough for me.
   - The transposed ui_matrix_df made the item-item matrix take a long time to compute as well.
- Creating the similarity matrices allowed me to create a function that let me pick and choose what model and similarity metric I wanted to use
    - By returning the user's predicted score for a movie and the top N similar movies, I was able to look and see if the scores hold up between rounds and test the similarly scored movies to see if they have similar scores.

- I think that adding in a similar genre score might make the recommendations even better
- Another potential improvement would be adding in themes of the movies like "dinosours" or "children's movie"